In [1]:
import os
import json
from pyi18next.i18next import I18next
from pyi18next.backends.fs import Backend
from pyi18next.utility import get_plural_func
from dotenv import load_dotenv
from pprint import pprint
import re


In [426]:
load_dotenv()


True

In [427]:
languages_str = os.getenv("LANGUAGES", "[]")
languages = json.loads(languages_str)

print(languages)

['es']


In [428]:
def traverse_namespaces(base_path: str, languages: list[str]):
	namespaces = set()

	for lng in languages:
		lng_path = os.path.join(base_path, lng)

		if os.path.isdir(lng_path):
			for root, _, files in os.walk(lng_path):
				for file in files:
					if file.endswith(".json"):
						full_path = os.path.join(root, file)
						
						rel_path = os.path.relpath(full_path, lng_path)

						namespace = os.path.splitext(rel_path)[0]
						namespace = namespace.replace(os.sep, "/")

						namespaces.add(namespace)

	return list(namespaces)

namespaces = traverse_namespaces("localization", languages)
print(namespaces)


['scene6/routeA/scene6LunchRouteA', 'scene6/routeB/scene6PoliceStationRouteB', 'computer/socialMediaScreen', 'scene6/routeB/scene6BedroomRouteB', 'scene6/routeA/scene6EndingRouteA', 'scene5/scene5Bedroom', 'scene5/scene5Livingroom', 'scene7/scene7Bedroom', 'scene1/scene1Lunch2', 'scene3/scene3Bedroom', 'scene6/routeA/scene6PortalRouteA', 'scene4/scene4Garage', 'menus/creditsScene', 'scene1/scene1Classroom', 'computer/loginScreen', 'menus/loginScene', 'scene2/scene2Break', 'scene1/scene1Bedroom2', 'scene2/scene2Bedroom', 'scene6/routeB/scene6LunchRouteB', 'scene1/scene1Bedroom1', 'scene6/routeA/scene6BedroomRouteA2', 'menus/titleScene', 'scene6/scene6Livingroom', 'scene1/scene1Break', 'scene4/scene4Frontyard', 'computer/captions', 'scene6/scene6Bedroom', 'generalDialogs', 'scene6/routeA/scene6BedroomRouteA1', 'transitions', 'deviceInfo', 'names', 'scene4/scene4Backyard', 'dialogManager', 'scene4/scene4Bedroom', 'scene6/routeB/scene6EndingRouteB', 'scene3/scene3Break', 'scene1/scene1Lunc

In [429]:
backend = Backend(name_mapping=lambda lng, ns: f"localization/{lng}/{ns}.json")

i18n = I18next(
	backend=backend,
	lng=languages,
	ns=namespaces,
)



In [430]:
pattern = re.compile(r'<([^>]+)>')

def expand_variants(text: str):
	matches = pattern.findall(text)
	if not matches:
		return [text]
	
	sentences = [text]
	
	for match in matches:
		variants = [v.strip() for v in match.split(',')][1:]
		new_sentences = []
		for sentence in sentences:
			for var in variants:
				# Remplaza la primera ocurrencia
				new_sentence = pattern.sub(var, sentence, count=1)
				new_sentences.append(new_sentence)
		sentences = new_sentences
	
	return sentences

text = "Igualmente, <player, encantado, encantada> de <jugar, conocerte, conocer> *sonríes*"
expand_variants(text)

['Igualmente, encantado de conocerte *sonríes*',
 'Igualmente, encantado de conocer *sonríes*',
 'Igualmente, encantada de conocerte *sonríes*',
 'Igualmente, encantada de conocer *sonríes*']

In [431]:
def process_data(data):
	if isinstance(data, str):
		data = data.encode("latin1").decode("utf-8")
		return expand_variants(data)
	elif isinstance(data, list):
		results = []
		for obj in data:
			expanded = process_data(obj)
			expanded = expanded if isinstance(expanded, list) else [expanded]
			results.extend(expanded)
		return results
	elif isinstance(data, dict):
		return {k: process_data(v) for k, v in data.items()}
	else:
		return data
	
texts = i18n.t("part2.thanks2.responses", ns="scene1/scene1Classroom", return_objects=True)
fixed_texts = process_data(texts)
pprint(fixed_texts)


None


In [432]:
# rules = "one: n is 1; other:"
rules = {
	"one": "n is 1",
	"other": ""
}

plural_func = get_plural_func(rules)

print(plural_func(1))
print(plural_func(3))


one
other


In [433]:
visited = set()

def build_full_id(language: str, filename: str, object_names: list[str], node_id: str):
	parts = [language, filename] + object_names + [node_id]
	return "_".join(parts)

def build_localization_id(object_names: list[str], node_id: str):
	parts = object_names + [node_id]
	return ".".join(parts)

def extract_next_nodes(node: dict, loc_id: str, language: str):
	next_nodes = []
	node_type = node.get("type")

	if "next" in node:
		next_nodes.append(node["next"])
			
	elif node_type == "choice" and "choices" in node:
		for choice in node["choices"]:
			if "next" in choice:
				next_nodes.append(choice["next"])
	
	elif node_type == "similarity":
		if "choices" in node:
			key = f"{loc_id}.responses"
			responses = i18n.t(key, ns="scene1/scene1Classroom", return_objects=True, lng=language)
			fixed_responses = process_data(responses)
			pprint(fixed_responses)
			
			for choice in node["choices"]:
				if "next" in choice:
					next_nodes.append(choice["next"])
					
		if "default" in node and "next" in node["default"]:
			next_nodes.append(node["default"]["next"])
	
	elif node_type == "condition" and "conditions" in node:
		for cond in node["conditions"]:
			if "next" in cond:
				next_nodes.append(cond["next"])
			
	return next_nodes

def dfs_traverse(language: str, filename: str, object_names: list[str], node_id: str, node_map: dict):
	full_id = build_full_id(language, filename, object_names, node_id)
	loc_id = build_localization_id(object_names, node_id)

	if full_id in visited:
		return

	visited.add(full_id)
	# print(full_id)

	node = node_map.get(node_id)
	if node:
		next_nodes = extract_next_nodes(node, loc_id, language)
		for next_node in next_nodes:
			dfs_traverse(language, filename, object_names, next_node, node_map)

def traverse_graph(language: str, filename: str, object_names: list[str], node_map: dict):
	if "root" in node_map:
		dfs_traverse(language, filename, object_names, "root", node_map)
	else:
		for sub_name, sub_map in node_map.items():
			new_object_names = object_names + [sub_name]
			traverse_graph(language, filename, new_object_names, sub_map)
	

In [434]:
def run(base_path: str, languages: list[str]):
	for root, _, files in os.walk(base_path):
		for file in files:
			if file.endswith(".json"):
				full_path = os.path.join(root, file)
				filename = os.path.splitext(os.path.basename(full_path))[0]

				with open(full_path, "r", encoding="utf-8") as f:
					data = json.load(f)

				for language in languages:
					if isinstance(data, dict) and "root" in data:
						traverse_graph(language, filename, [], data)

					elif isinstance(data, dict):
						for object_name, node_map in data.items():
							traverse_graph(language, filename, [object_name], node_map)

	print(f"Total visited nodes: {len(visited)}")


languages = ["es"]
run("localization/structure", languages)


[{'text': ['Igualmente, encantado de conocerte *sonríes*.',
           'Igualmente, encantada de conocerte *sonríes*.',
           'El gusto es mío, amigo *asientes con la cabeza*.',
           'El gusto es mío, amiga *asientes con la cabeza*.',
           'Un placer conocerte también, encantado *sonríes levemente*.',
           'Un placer conocerte también, encantada *sonríes levemente*.']},
 {'text': ['Gracias, si necesito algo ya te iré diciendo.',
           'De acuerdo, ya te diré si necesito algo.',
           'Perfecto, muchas gracias por avisar.']},
 {'text': ['... Ah, sí, hola.', 'Eh... claro, sí.', 'Ah, hola.']}]
Total visited nodes: 670
